0) Setup + Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_auc_score, auc
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, average_precision_score
from sklearn.svm import OneClassSVM
from sklearn.ensemble import HistGradientBoostingClassifier

np.random.seed(560)
SEED=560 #random seed

1) Load Data

In [ ]:
from google.colab import files
uploaded = files.upload()



Saving balanced_500k.log.gz to balanced_500k.log.gz


In [ ]:
df = pd.read_csv(
    "balanced_500k.log.gz",
    sep=r"\s+",
    comment="#",
    header=None,
    engine="python"
)

df.columns = [
    "ts", "uid", "id.orig_h", "id.orig_p", "id.resp_h", "id.resp_p",
    "proto", "service", "duration", "orig_bytes", "resp_bytes",
    "conn_state", "local_orig", "local_resp", "missed_bytes",
    "history", "orig_pkts", "orig_ip_bytes", "resp_pkts",
    "resp_ip_bytes", "tunnel_parents", "label", "detailed_label"
]

df.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed_label
0,1.547089e+09,CnMFIA2nU9Sc44Hlk2,192.168.1.194,51135,184.219.175.83,22,tcp,-,-,-,...,-,0,S,1,40,0,0,-,Malicious,PartOfAHorizontalPortScan
1,1.545378e+09,CcLuxP3p0sXi5OJula,192.168.1.197,43746,161.36.170.197,80,tcp,-,-,-,...,-,0,S,1,40,0,0,-,Malicious,PartOfAHorizontalPortScan
2,1.545367e+09,Ckubnx10ByDlZ33Or9,192.168.1.197,44002,197.125.59.2,52869,tcp,-,-,-,...,-,0,S,1,40,0,0,-,Malicious,PartOfAHorizontalPortScan
3,1.545386e+09,C6Ricd1EYUC4po0dGc,192.168.1.197,38370,197.147.38.29,37215,tcp,-,-,-,...,-,0,S,1,40,0,0,-,Malicious,Okiru
4,1.536303e+09,CfjxxD1aUwHEnW0Hzj,192.168.100.111,17832,197.59.229.170,37215,tcp,-,0.000003,0,...,-,0,S,2,80,0,0,-,Malicious,Okiru


In [ ]:
print(df.columns.tolist())
df["label"] = df["label"].str.strip().str.replace("-", "")
df["label"] = df["label"].apply(lambda x: 0 if x == "Benign" else 1)
df = df.drop(columns=[
    "uid",
    "id.orig_h",
    "id.resp_h",
    "tunnel_parents",
    "detailed_label",
    "ts",
    "id.orig_p"
], errors="ignore")

['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes', 'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'tunnel_parents', 'label', 'detailed_label']


In [ ]:
df_sub = df.sample(n=100000, random_state=560)

In [ ]:
X = df_sub.drop(columns=['label'])
y = df_sub['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=560, stratify=y)  #splitting set in 80/20 split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=560, stratify=y_train) #splitting 80 from last split into 60/20

print("Training values: ", len(X_train))
print("%(0): ", ((y_train == 0).mean() * 100).round(2))
print("%(1): ", ((y_train == 1).mean() * 100).round(2))
print("\nValidation values: ", len(X_val))
print("%(0): ", ((y_val == 0).mean() * 100).round(2))
print("%(1): ", ((y_val == 1).mean() * 100).round(2))
print("\nTesting values: ", len(X_test))
print("%(0): ", ((y_test == 0).mean() * 100).round(2))
print("%(1): ", ((y_test == 1).mean() * 100).round(2))

Training values:  60000
%(0):  50.07
%(1):  49.93

Validation values:  20000
%(0):  50.08
%(1):  49.92

Testing values:  20000
%(0):  50.07
%(1):  49.93


In [ ]:
numeric = X_train.select_dtypes(include=['number']).columns.tolist()  #grabbing columns only with numeric values
categorical = X_train.select_dtypes(include=['object']).columns.tolist()  #grabbing columns with only categorical columns

imputer = SimpleImputer(strategy='median')  #imputing numeric columns using median imputation
scaler = StandardScaler() #performing standard scaling on numeric columns

num_imputer = imputer
scaler = scaler

X_train[numeric] = imputer.fit_transform(X_train[numeric])
X_train[numeric] = scaler.fit_transform(X_train[numeric])
X_val[numeric] = imputer.transform(X_val[numeric])
X_val[numeric] = scaler.transform(X_val[numeric])
X_test[numeric] = imputer.transform(X_test[numeric])
X_test[numeric] = scaler.transform(X_test[numeric])

imputer = SimpleImputer(strategy='most_frequent') #using most frequency imputation on categorical columns

cat_imputer = imputer

X_train[categorical] = imputer.fit_transform(X_train[categorical])
X_train = pd.get_dummies(X_train,columns=categorical) #one hot encoding categorical columns
X_val[categorical] = imputer.transform(X_val[categorical])
X_val = pd.get_dummies(X_val,columns=categorical)
X_test[categorical] = imputer.transform(X_test[categorical])
X_test = pd.get_dummies(X_test,columns=categorical)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Training: ",X_train.shape)  #final matrix shapes
print("Validation: ",X_val.shape)
print("Testing: ",X_test.shape)

print(f"Training{len(X_train)}, {y_train.value_counts(normalize=True)*100}")
print(f"Validation{len(X_val)}, {y_val.value_counts(normalize=True)*100}")
print(f"Testing{len(X_test)}, {y_test.value_counts(normalize=True)*100}")

Training:  (60000, 4265)
Validation:  (20000, 4265)
Testing:  (20000, 4265)
Training60000, label
0    50.071667
1    49.928333
Name: proportion, dtype: float64
Validation20000, label
0    50.075
1    49.925
Name: proportion, dtype: float64
Testing20000, label
0    50.07
1    49.93
Name: proportion, dtype: float64


Decision Tree

In [ ]:
best_model_global = None
best_f1_global = 0

In [ ]:
def f1sweeper(ytrue,scores): # finding best f1
  thresholds=np.linspace(scores.min(),scores.max(),100)
  bestt=0
  bestf1=0

  for t in thresholds:
    ypred=[1 if x>t else 0 for x in scores]
    f1=f1_score(ytrue,ypred)

    if f1>bestf1:
      bestf1=f1
      bestt=t

  return bestt, bestf1

def metrics(ytrue,scores,threshold):
  ypred=[1 if x>threshold else 0 for x in scores]
  rocauc=roc_auc_score(ytrue,scores)
  prauc=average_precision_score(ytrue,scores)
  precision=precision_score(ytrue,ypred)
  recall=recall_score(ytrue,ypred)
  f1=f1_score(ytrue,ypred)
  return {
      'rocauc': rocauc,
      'prauc': prauc,
      'precision': precision,
      'recall': recall,
      'f1': f1
  }


results = []
def recordresults(modelname,ytrue,scores,threshold):

  numbers=metrics(ytrue,scores,threshold)

  results.append({
      'model':modelname,
      'precision':numbers['precision'],
      'recall':numbers['recall'],
      'f1':numbers['f1'],
      'rocauc':numbers['rocauc'],
      'prauc':numbers['prauc']
  })
  return results

DTresults = []
def recordresultsDT(depth,minleaf,minsplits,accuracy,precision,recall,f1):

  DTresults.append({
      'depth':depth,
      'minleaf':minleaf,
      'minsplits':minsplits,
      'accuracy':accuracy,
      'precision':precision,
      'recall':recall,
      'f1':f1
  })
  return DTresults

SVMresults = []
def recordresultsSVM(nu,gamma,accuracy,precision,recall,f1):

  SVMresults.append({
      'nu':nu,
      'gamma':gamma,
      'accuracy':accuracy,
      'precision':precision,
      'recall':recall,
      'f1':f1
  })
  return SVMresults

HISTresults = []
def recordresultsHIST(learning_rate,max_depth,min_samples_leaf,accuracy,precision,recall,f1):

  HISTresults.append({
      'learning_rate':learning_rate,
      'max_depth':max_depth,
      'min_samples_leaf':min_samples_leaf,
      'accuracy':accuracy,
      'precision':precision,
      'recall':recall,
      'f1':f1
  })
  return HISTresults




In [ ]:
depth=[4,6,8]
minleaf=[1,5,10]
minsplits=[20, 50, 100]

best_f1=0
best_parameters={}
best_model=None
best_t=0
bestvalprob=0

for d in depth:
  for m in minleaf:
    for s in minsplits:
      model = DecisionTreeClassifier(max_depth=d,min_samples_leaf=m,min_samples_split=s,random_state=SEED)
      model.fit(X_train, y_train)

      val_prob=model.predict_proba(X_val)[:,1]
      t,f1=f1sweeper(y_val,val_prob)
      val_preds = (val_prob >= t).astype(int)
      acc = accuracy_score(y_val, val_preds)
      prec = precision_score(y_val, val_preds)
      rec = recall_score(y_val, val_preds)
      recordresultsDT(d,m,s,acc,prec,rec,f1)

      if f1>best_f1:
        best_f1=f1
        best_parameters={'depth':d,'minleaf':m,'minsplits':s}
        best_model=model
        best_t=t

print(f"Best parameters: {best_parameters}")
print(f"Best F1: {best_f1}")

if best_f1 > best_f1_global:
  best_f1_global = best_f1
  best_model_global = best_model
  best_threshold_global = best_t

df_results = pd.DataFrame(DTresults)
DT_results = df_results.sort_values(by="f1", ascending=False)
print(DT_results)
final_test_prob=best_model.predict_proba(X_test)[:,1]
recordresults('Model #1 - Decision Tree',y_test,final_test_prob,best_t)



Best parameters: {'depth': 8, 'minleaf': 10, 'minsplits': 100}
Best F1: 0.9914184755174155
    depth  minleaf  minsplits  accuracy  precision    recall        f1
26      8       10        100   0.99150   0.999491  0.983475  0.991418
18      8        1         20   0.99145   0.999085  0.983776  0.991371
19      8        1         50   0.99145   0.999085  0.983776  0.991371
23      8        5        100   0.99145   0.999085  0.983776  0.991371
20      8        1        100   0.99145   0.999085  0.983776  0.991371
24      8       10         20   0.99145   0.999491  0.983375  0.991368
25      8       10         50   0.99145   0.999491  0.983375  0.991368
22      8        5         50   0.99140   0.998983  0.983776  0.991321
21      8        5         20   0.99140   0.998983  0.983776  0.991321
10      6        1         50   0.98940   0.999183  0.979569  0.989279
9       6        1         20   0.98940   0.999183  0.979569  0.989279
11      6        1        100   0.98940   0.999183  0.979

[{'model': 'Model #1 - Decision Tree',
  'precision': 0.998883588754694,
  'recall': 0.985579811736431,
  'f1': 0.9921871062049499,
  'rocauc': np.float64(0.9934946672495479),
  'prauc': np.float64(0.9933428273550872)}]

In [ ]:
results_df = pd.DataFrame(results)
print(results_df)

                      model  precision   recall        f1    rocauc     prauc
0  Model #1 - Decision Tree   0.998884  0.98558  0.992187  0.993495  0.993343


One Class SVM

In [ ]:
X_train_OC=X_train[y_train==0]
print(X_train_OC.shape)

(30043, 4265)


In [ ]:
nu=[.01,.05,.1]
gamma=['scale','auto',.1]

best_f1=0
best_parameters={}
best_model=None
best_t=0
best_val_prob=0

for n in nu:
  for g in gamma:
    OCSVM=OneClassSVM(nu=n,gamma=g)
    OCSVM.fit(X_train_OC)

    val_prob=-OCSVM.decision_function(X_val)
    t,f1=f1sweeper(y_val,val_prob)
    val_preds = (val_prob >= t).astype(int)
    acc = accuracy_score(y_val, val_preds)
    prec = precision_score(y_val, val_preds)
    rec = recall_score(y_val, val_preds)
    recordresultsSVM(n,g,acc,prec,rec,f1)

    if f1 > best_f1:
      best_f1=f1
      best_t=t
      best_parameters={'Nu':n,'gamma':g}
      best_model=OCSVM
      best_val_prob=val_prob

if best_f1 > best_f1_global:
  best_f1_global = best_f1
  best_model_global = best_model
  best_threshold_global = best_t

df_results = pd.DataFrame(SVMresults)
SVM_results = df_results.sort_values(by="f1", ascending=False)
print(SVM_results)
final_test_prob=best_model.decision_function(X_test)
recordresults('Model #2 - OC-SVM',y_test,final_test_prob,best_t)



     nu  gamma  accuracy  precision    recall        f1
5  0.05    0.1   0.81030   0.969656  0.640060  0.771115
6  0.10  scale   0.80925   0.969559  0.637957  0.769556
8  0.10    0.1   0.61765   0.566846  0.992789  0.721654
3  0.05  scale   0.61980   0.571369  0.954532  0.714843
0  0.01  scale   0.55885   0.530934  0.998698  0.693294
2  0.01    0.1   0.55830   0.530604  0.999299  0.693157
7  0.10   auto   0.49925   0.499250  1.000000  0.666044
1  0.01   auto   0.49925   0.499250  1.000000  0.666022
4  0.05   auto   0.49925   0.499250  1.000000  0.665955


Histogram-Based Gradient Boosting

In [ ]:
learning_rate=[.05, 0.1]
max_depth=[4, 8]
min_samples_leaf=[20, 100]

best_f1=0
best_parameters={}
best_model=None
best_t=0
best_val_prob=0

for l in learning_rate:
    for d in max_depth:
      for s in min_samples_leaf:
        model = HistGradientBoostingClassifier(learning_rate=l,max_depth=d,min_samples_leaf=s)
        model.fit(X_train, y_train)

        val_prob=model.predict_proba(X_val)[:,1]
        t,f1=f1sweeper(y_val,val_prob)
        val_preds = (val_prob >= t).astype(int)
        acc = accuracy_score(y_val, val_preds)
        prec = precision_score(y_val, val_preds)
        rec = recall_score(y_val, val_preds)
        recordresultsHIST(l,d,s,acc,prec,rec,f1)

        if f1 > best_f1:
          best_f1=f1
          best_t=t
          best_parameters={'learning_rate':l,'max_depth':d,'min_samples_leaf':s}
          best_model=model
          best_val_prob=val_prob

if best_f1 > best_f1_global:
  best_f1_global = best_f1
  best_model_global = best_model
  best_threshold_global = best_t

df_results = pd.DataFrame(HISTresults)
HIST_results = df_results.sort_values(by="f1", ascending=False)
print(HIST_results)

final_test_prob=best_model.predict_proba(X_test)[:,1]
recordresults('Model #3 - Histogram-Based Gradient Boosting',y_test,final_test_prob,best_t)


   learning_rate  max_depth  min_samples_leaf  accuracy  precision    recall  \
6           0.10          8                20   0.99255   0.999289  0.985779   
2           0.05          8                20   0.99245   0.999492  0.985378   
7           0.10          8               100   0.99230   0.998782  0.985779   
4           0.10          4                20   0.99225   1.000000  0.984477   
5           0.10          4               100   0.99200   0.998782  0.985178   
0           0.05          4                20   0.99190   0.998174  0.985578   
3           0.05          8               100   0.99175   0.999085  0.984377   
1           0.05          4               100   0.99110   0.997363  0.984777   

         f1  
6  0.992488  
2  0.992385  
7  0.992238  
4  0.992178  
5  0.991933  
0  0.991836  
3  0.991676  
1  0.991030  


[{'model': 'Model #3 - Histogram-Based Gradient Boosting',
  'precision': 0.9980780902286062,
  'recall': 0.9880833166433006,
  'f1': 0.9930555555555556,
  'rocauc': np.float64(0.9958228318127503),
  'prauc': np.float64(0.9953080786121756)}]

In [ ]:
print(DT_results)
print(SVM_results)
print(HIST_results)